# Model Training

Weighted BCE + Adam, batch size 1 (one exam), 50 epochs with early stopping, 2D slice augmentation.

In [1]:
import os
import copy
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import torchvision
import torchvision.models as tvm
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset, DataLoader, Subset

SEED = 42
IMG_DIR = 'images'
PLANES = ('axial', 'coronal', 'sagittal')
TASKS = ('abnormal', 'acl', 'meniscus')
TARGET_SLICES = 32
LR = 1e-3
EPOCHS = 50
BATCH_SIZE = 1
PATIENCE = 5
BACKBONE = 'resnet18'
OUT_DIR = 'outputs'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('device:', DEVICE, '| batch size:', BATCH_SIZE, '| epochs:', EPOCHS, '| patience:', PATIENCE)

device: cuda | batch size: 1 | epochs: 50 | patience: 5


Data: same preprocessing as `Data_processing.ipynb` (standardize + resample to 32 slices).

In [2]:
def standardize(arr):
    arr = arr.astype(np.float32)
    return (arr - arr.mean()) / (arr.std() + 1e-8)


def preprocess(arr, target=TARGET_SLICES):
    arr = standardize(arr)
    t = torch.from_numpy(np.ascontiguousarray(arr))[None, None]
    arr = F.interpolate(t, size=(target, arr.shape[1], arr.shape[2]),
                        mode='trilinear', align_corners=False)[0, 0].numpy()
    return torch.from_numpy(np.stack([arr] * 3))  # (3, D, H, W)

In [3]:
def _slice_batch(vol):
    d, h, w = vol.shape[1:]
    return vol.permute(1, 0, 2, 3).reshape(-1, 1, h, w)


def _unslice_batch(imgs, c, d):
    h, w = imgs.shape[-2:]
    return imgs.reshape(d, c, h, w).permute(1, 0, 2, 3)


class RandomHFlip:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, vol):
        return torch.flip(vol, dims=[-1]) if torch.rand(1).item() < self.p else vol


class RandomRotation:
    def __init__(self, degrees=15.0, p=0.5):
        self.degrees, self.p = degrees, p

    def __call__(self, vol):
        if torch.rand(1).item() >= self.p:
            return vol
        ang = (torch.rand(1).item() * 2 - 1) * self.degrees
        imgs = TF.rotate(_slice_batch(vol), ang, interpolation=InterpolationMode.BILINEAR, fill=0.0)
        return _unslice_batch(imgs, vol.shape[0], vol.shape[1])


class RandomTranslation:
    def __init__(self, fraction=0.1, p=0.5):
        self.fraction, self.p = fraction, p

    def __call__(self, vol):
        if torch.rand(1).item() >= self.p:
            return vol
        dx = int(round((torch.rand(1).item() * 2 - 1) * self.fraction * vol.shape[-1]))
        dy = int(round((torch.rand(1).item() * 2 - 1) * self.fraction * vol.shape[-2]))
        imgs = TF.affine(_slice_batch(vol), angle=0, translate=(dx, dy), scale=1.0,
                         shear=[0.0, 0.0], interpolation=InterpolationMode.BILINEAR, fill=0.0)
        return _unslice_batch(imgs, vol.shape[0], vol.shape[1])


class VolumeCompose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, vol):
        for t in self.transforms:
            vol = t(vol)
        return vol


TRAIN_TRANSFORMS = VolumeCompose([RandomHFlip(0.5), RandomRotation(15, 0.5), RandomTranslation(0.1, 0.5)])

In [4]:
class MRNetDataset(Dataset):
    def __init__(self, root=IMG_DIR, task='abnormal', split='train', plane='sagittal',
                 transform=None):
        self.folder = os.path.join(root, split, plane)
        df = pd.read_csv(os.path.join(root, f'{split}-{task}.csv'),
                         header=None, names=['id', 'label'])
        df['id'] = df['id'].astype(int).astype(str).str.zfill(4)
        self.ids = df['id'].tolist()
        self.labels = df['label'].astype(np.float32).tolist()
        self.transform = transform
        pos, neg = sum(self.labels), len(self.labels) - sum(self.labels)
        self.weights = torch.tensor([1.0, neg / max(pos, 1.0)])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        vol = preprocess(np.load(os.path.join(self.folder, self.ids[idx] + '.npy')))
        if self.transform is not None:
            vol = self.transform(vol)
        return vol, torch.tensor(self.labels[idx])

Model: pretrained 2D backbone, slice features max+logsumexp pooled (original MRNet aggregation).

In [5]:
class MRNet2D(nn.Module):
    def __init__(self, backbone='resnet18', pretrained=True, freeze_backbone=False):
        super().__init__()
        if backbone == 'resnet18':
            m = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            m.fc = nn.Identity()
            self.backbone, self.feat_dim = nn.Sequential(*list(m.children())), 512
        elif backbone == 'alexnet':
            m = tvm.alexnet(weights=tvm.AlexNet_Weights.IMAGENET1K_V1 if pretrained else None)
            self.backbone, self.feat_dim = nn.Sequential(m.features, nn.AdaptiveAvgPool2d(1), nn.Flatten()), 256
        else:
            raise ValueError(backbone)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.fc = nn.Linear(2 * self.feat_dim, 1)

    def forward(self, x):                     # (B, 3, D, H, W)
        b, c, d, h, w = x.shape
        f = self.backbone(x.permute(0, 2, 1, 3, 4).reshape(b * d, c, h, w)).view(b, d, self.feat_dim)
        return self.fc(torch.cat([f.max(1).values, f.logsumexp(1)], dim=1))


def build_model(backbone=BACKBONE, pretrained=True):
    return MRNet2D(backbone=backbone, pretrained=pretrained).to(DEVICE)

In [6]:
model = build_model()
print('params_M:', round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
      '| trainable_M:', round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 2))

train_probe = MRNetDataset(task='acl', split='train', plane='sagittal', transform=TRAIN_TRANSFORMS)
print('train pos_weight:', train_probe.weights.tolist(), f'| positives: {sum(train_probe.labels):.0f}/{len(train_probe)}')
vol, y = train_probe[0]
print('augmented sample:', tuple(vol.shape), 'label:', y.item(),
      '| finite:', bool(torch.isfinite(vol).all()))

params_M: 11.18 | trainable_M: 11.18
train pos_weight: [1.0, 4.432692527770996] | positives: 208/1130
augmented sample: (3, 32, 256, 256) label: 0.0 | finite: True


Training: `BCEWithLogitsLoss(pos_weight)` + Adam; early stopping on validation loss.

In [7]:
def auc_score(labels, scores):
    y = np.asarray(labels, dtype=np.float64)
    r = pd.Series(np.asarray(scores, dtype=np.float64)).rank(method='average').values
    pos, neg = y.sum(), len(y) - y.sum()
    if pos == 0 or neg == 0:
        return float('nan')
    return float((r[y == 1].sum() - pos * (pos + 1) / 2) / (pos * neg))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.float().to(DEVICE)
        optimizer.zero_grad()
        logits = model(x).view(-1)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
        n += len(y)
    return total / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total, n, ys, ps = 0.0, 0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.float().to(DEVICE)
        logits = model(x).view(-1)
        total += criterion(logits, y).item() * len(y)
        n += len(y)
        ys.append(y.cpu())
        ps.append(torch.sigmoid(logits).cpu())
    y = torch.cat(ys)
    p = torch.cat(ps)
    return total / max(n, 1), auc_score(y, p), float((((p >= 0.5).float() == y).float().mean()))

In [8]:
def train_model(task, plane, backbone=BACKBONE, epochs=None, patience=None,
                limit_train=None, limit_val=None, save_name=None, pretrained=True):
    epochs = epochs or EPOCHS
    patience = patience or PATIENCE
    train_full = MRNetDataset(task=task, split='train', plane=plane, transform=TRAIN_TRANSFORMS)
    valid_ds = MRNetDataset(task=task, split='valid', plane=plane)
    pos_weight = train_full.weights[1].to(DEVICE)
    if limit_train:
        idx = np.random.default_rng(SEED).choice(len(train_full), size=min(limit_train, len(train_full)), replace=False)
        train_ds = Subset(train_full, idx.tolist())
    else:
        train_ds = train_full
    if limit_val:
        idx = np.random.default_rng(SEED).choice(len(valid_ds), size=min(limit_val, len(valid_ds)), replace=False)
        valid_ds = Subset(valid_ds, idx.tolist())
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_model(backbone, pretrained)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_acc': []}
    best_val, best_state, wait = float('inf'), None, 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tl = train_one_epoch(model, train_loader, criterion, optimizer)
        vl, va, vac = evaluate(model, valid_loader, criterion)
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['val_auc'].append(va)
        history['val_acc'].append(vac)
        flag = ''
        if vl < best_val:
            best_val, best_state, wait, flag = vl, copy.deepcopy(model.state_dict()), 0, ' *best*'
        else:
            wait += 1
        print(f'epoch {epoch:3d}/{epochs} | train {tl:.4f} | val {vl:.4f} | auc {va:.3f} '
              f'| acc {vac:.3f} | {time.time() - t0:.1f}s{flag}', flush=True)
        if wait >= patience:
            print(f'early stopping at epoch {epoch} (no improvement for {patience} epochs)')
            break

    model.load_state_dict(best_state)
    save_name = save_name or f'best_{task}_{plane}.pth'
    torch.save({'task': task, 'plane': plane, 'epoch': len(history['train_loss']),
                'val_loss': best_val, 'model_state': best_state},
               os.path.join(OUT_DIR, save_name))
    print(f'saved {save_name} | best val_loss {best_val:.4f}')
    return model, history

In [9]:
RUN_DEMO = False


def train_all(epochs=None):
    for plane in PLANES:
        for task in TASKS:
            train_model(task, plane, epochs=epochs)


if RUN_DEMO:
    _, hist = train_model('acl', 'sagittal', epochs=2, limit_train=8, limit_val=8)
else:
    print("set RUN_DEMO=True, or call train_model(task, plane) / train_all() to start training")

set RUN_DEMO=True, or call train_model(task, plane) / train_all() to start training
